# Lesson 10 — UART 是字节流，不是消息队列 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 10-uart-terminal solution exercise pass
#pragma once
#include "academy/device.hpp"
#include <string>
namespace academy {
    class Controller {
    public:
        explicit Controller(Device&);
        void tick();

    private:
        Device& board_;
        std::string buffer_;
    };
}

namespace academy {
    Controller::Controller(Device& b) : board_(b) {
    }
    void Controller::tick() {
        if (!board_.input.connected) {
            board_.output.error = true;
            board_.output.state = "disconnected";
            return;
        }
        board_.output.error = false;
        buffer_ += board_.input.text;
        std::size_t end;
        while ((end = buffer_.find('\n')) != std::string::npos) {
            std::string f = buffer_.substr(0, end);
            buffer_.erase(0, end + 1);
            if (f == "PING")
                board_.output.outgoing += "PONG\n";
            else if (f == "LED:1") {
                board_.output.led = true;
                board_.output.outgoing += "OK\n";
            } else if (f == "LED:0") {
                board_.output.led = false;
                board_.output.outgoing += "OK\n";
            } else
                board_.output.outgoing += "ERR\n";
        }
        board_.output.state = "uart-ready";
    }
}
